# Linear Regression Training — Normal Equation

আগের lesson-এ আমরা জানতাম:

$$
\hat y=Xw
$$

কিন্তু `w` কোথা থেকে আসবে?

এই lesson-এর পুরো cycle:

1. কেন `X`-এর inverse নেওয়া যায় না
2. কেন $X^TX$ square matrix হয়
3. Gram Matrix-এর inverse ব্যবহার করে Normal Equation
4. NumPy-তে পুরো implementation

Final formula:

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

## 1. কেন `X`-এর inverse নেওয়া যায় না?

ekta matrix ke inverse korte hole take obosshoi square matrix hote hobe.

আমাদের ideal equation:

$$
Xw=y
$$

যদি `X` square এবং invertible হতো:

$$
X^{-1}Xw=X^{-1}y
$$

তাই:

$$
w=X^{-1}y
$$

কিন্তু real dataset-এ `X` সাধারণত **rectangular**।

ধরি:

$$
X=9\times3
$$

মানে:
- 9 rows = 9 training examples
- 3 columns = 3 features

যেহেতু:

$$
9\neq3
$$

তাই `X` square নয় এবং `X^{-1}` নেওয়া যায় না।

অর্থাৎ সরাসরি `X` inverse করে `w` বের করা যাবে না।

## 2. কেন $X^TX$ square matrix হয়?

যদি:

$$
X=(m\times n)
$$

তাহলে transpose:

$$
X^T=(n\times m)
$$

এখন:

$$
X^TX=(n\times m)(m\times n)
$$

মাঝের `m` match করে, তাই multiplication valid।

Result:

$$
\boxed{X^TX=n\times n}
$$

### Example

যদি:

$$
X=9\times3
$$

তাহলে:

$$
X^T=3\times9
$$

এবং:

$$
X^TX=(3\times9)(9\times3)=3\times3
$$

এই $X^TX$-কেই **Gram Matrix** বলা হয়:

$$
\boxed{G=X^TX}
$$

## 3. Gram Matrix থেকে Normal Equation

আমাদের goal:

$$
Xw\approx y
$$

দুই পাশে $X^T$ দিয়ে multiply করি:

$$
X^TXw=X^Ty
$$

এখন $X^TX$ square। তাই inverse নেওয়া সম্ভব হলে:

$$
(X^TX)^{-1}X^TXw=(X^TX)^{-1}X^Ty
$$

কারণ:

$$
(X^TX)^{-1}(X^TX)=I
$$

তাই:

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

এটাই **Normal Equation**।

এই `w` এমন weights দেয় যাতে:

$$
Xw
$$

actual `y`-এর যতটা সম্ভব কাছাকাছি হয়।

### Bias

Bias $w_0$-কে matrix calculation-এর মধ্যে রাখার জন্য `X`-এর সামনে এক column of `1` যোগ করি:

```text
[1, x1, x2, x3]
```

তাহলে:

$$
1\times w_0+x_1w_1+x_2w_2+x_3w_3
$$

অর্থাৎ bias-ও একই calculation-এর অংশ হয়ে যায়।

# 4. NumPy Implementation

পুরো flow:

```text
X, y
 ↓
Add bias column
 ↓
X.T @ X
 ↓
Inverse
 ↓
X.T @ y
 ↓
w = (X.T @ X)^(-1) @ X.T @ y
 ↓
Predictions = X @ w
```

In [ ]:
import numpy as np

# 9 cars, 3 features
X = np.array([
    [148, 24, 1385],
    [132, 25, 2031],
    [453, 11, 86],
    [158, 24, 185],
    [172, 25, 201],
    [413, 11, 86],
    [38,  54, 185],
    [142, 25, 431],
    [453, 31, 86],
], dtype=float)

# Actual target values
y = np.array([100, 200, 150, 250, 100, 200, 150, 250, 120], dtype=float)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Add the bias column of 1s

ones = np.ones(X.shape[0])
X = np.column_stack([ones, X])

print("X shape after adding bias:", X.shape)
print(X)


In [ ]:
# Normal Equation

XTX = X.T.dot(X)                  # Gram Matrix
XTX_inv = np.linalg.inv(XTX)      # Gram Matrix inverse
XTy = X.T.dot(y)

w_full = XTX_inv.dot(XTy)

# First value = bias, rest = feature weights
w0 = w_full[0]
w = w_full[1:]

print("Bias w0:", w0)
print("Feature weights w:", w)


In [ ]:
# Sanity check: A matrix × its inverse ≈ Identity

print(XTX.dot(XTX_inv).round(1))


In [ ]:
# Prediction

predictions = X.dot(w_full)

print("Actual y:")
print(y)

print("\nPredictions:")
print(predictions)


## Final Training Function

উপরের সব step function-এর মধ্যে রাখা যায়:

```python
w0, w = train_linear_regression(X, y)
```

Function:
- bias column add করবে
- Gram Matrix তৈরি করবে
- inverse বের করবে
- Normal Equation দিয়ে weights বের করবে
- bias এবং feature weights আলাদা করে return করবে

In [ ]:
def train_linear_regression(X, y):
    # Add bias column
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    # Normal Equation
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    XTy = X.T.dot(y)

    w_full = XTX_inv.dot(XTy)

    return w_full[0], w_full[1:]


w0, w = train_linear_regression(X[:, 1:], y)

print("w0:", w0)
print("w :", w)


# Final Summary

### 1. `X` inverse করা যায় না

কারণ `X` সাধারণত rectangular:

$$
X=m\times n,\quad m\neq n
$$

### 2. `X.T @ X` square

$$
X^TX=(n\times m)(m\times n)=n\times n
$$

তাই এটি square **Gram Matrix**।

### 3. Normal Equation

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

### 4. Prediction

$$
\boxed{\hat y=Xw}
$$

### পুরো cycle

```text
X + y
 ↓
Normal Equation
 ↓
w
 ↓
X @ w
 ↓
ŷ
```

**Training = `w` খুঁজে বের করা**

**Prediction = `w` ব্যবহার করে `ŷ` বের করা**